# TUGAS 1 KECERDASAN ARTIFISIAL UNTUK GEOFISIKA
## Perancangan Prompt AI dan Penyelesaian Kasus Geofisika Menggunakan Python

**Nama:** A. Dwita Ulya Azizah  
**NIM:** H061231050  
**Kelas:** Geofisika B

---

### Deskripsi
Notebook ini menunjukkan perancangan prompt AI menggunakan kerangka **R-T-C-I-K-O-Q-V** serta penerapan kode Python untuk menyelesaikan lima kasus geofisika: seismik refraksi, geolistrik resistivitas, geomagnetik, gravitasi, dan magnetotelurik (MT).

**Catatan penting:** data yang digunakan adalah data simulasi untuk pembelajaran, bukan data pengukuran lapangan. Hasil interpretasi bersifat indikatif dan harus diverifikasi dengan data lapangan, informasi geologi, serta pemodelan yang sesuai.

## 1. Persiapan Lingkungan Python
Kode berikut mengimpor pustaka yang umum tersedia di Google Colab/Jupyter. Jika pustaka belum tersedia, instalasi dapat dilakukan sesuai lingkungan yang digunakan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=3, suppress=True)

print("Pustaka berhasil dimuat.")

## 2. Kerangka Perancangan Prompt AI (R-T-C-I-K-O-Q-V)

- **R — Role:** peran AI.
- **T — Task:** tugas yang harus dikerjakan.
- **C — Context:** latar belakang kasus.
- **I — Input:** data yang tersedia.
- **K — Constraints:** batasan dan asumsi.
- **O — Output:** bentuk keluaran.
- **Q — Quality:** standar kualitas.
- **V — Verification:** langkah verifikasi hasil.

Prompt harus menyatakan data, satuan, asumsi, metode perhitungan, keluaran yang diminta, dan cara memeriksa hasil. AI tidak boleh menyatakan data simulasi sebagai hasil pengukuran nyata.

# KASUS 1 — SEISMIK REFRAKSI
## Estimasi kecepatan lapisan dan kedalaman bidang refraktor

### Rumusan kasus
Survei seismik refraksi menghasilkan waktu tempuh gelombang pada beberapa jarak geofon. Untuk model dua lapisan dengan lapisan bawah lebih cepat, waktu tempuh langsung dapat ditulis `t = x/V1`, sedangkan waktu gelombang refraksi kepala dapat ditulis `t = ti + x/V2`. Kemiringan garis waktu-tempuh digunakan untuk mengestimasi kecepatan, sementara intersep refraksi digunakan untuk menghitung kedalaman antarmuka pada asumsi model dua lapisan horizontal.

### Prompt AI (R-T-C-I-K-O-Q-V)

**R:** Anda adalah asisten geofisika yang memahami seismik refraksi.  
**T:** Estimasikan kecepatan lapisan pertama dan kedua serta kedalaman antarmuka.  
**C:** Model terdiri dari dua lapisan horizontal; kecepatan lapisan kedua lebih besar daripada lapisan pertama.  
**I:** Data jarak geofon dan waktu tiba gelombang pertama (simulasi).  
**K:** Gunakan satuan SI; nyatakan asumsi; jangan menganggap hasil sebagai kondisi lapangan nyata.  
**O:** Tampilkan tabel data, kecepatan, kedalaman antarmuka, dan grafik waktu-tempuh.  
**Q:** Periksa urutan kecepatan dan konsistensi satuan.  
**V:** Bandingkan kurva hasil model dengan data simulasi dan jelaskan keterbatasan model.

In [ ]:
# Data simulasi: dua segmen waktu-tempuh dengan sedikit noise
rng = np.random.default_rng(12)
x_m = np.arange(10, 101, 10, dtype=float)
V1_true, V2_true, h_true = 500.0, 1500.0, 8.0  # m/s, m/s, m
ti_true = 2 * h_true * np.sqrt(V2_true**2 - V1_true**2) / (V1_true * V2_true)

t_direct = x_m / V1_true
t_head = ti_true + x_m / V2_true
t_first_clean = np.minimum(t_direct, t_head)
t_obs = t_first_clean + rng.normal(0, 0.0015, size=len(x_m))
t_obs = np.maximum(t_obs, 0.001)

# Estimasi V1 dari bagian jarak pendek dan V2/intercept dari bagian jarak jauh
short = x_m <= 30
long = x_m >= 50
slope1, intercept1 = np.polyfit(x_m[short], t_obs[short], 1)
slope2, intercept2 = np.polyfit(x_m[long], t_obs[long], 1)
V1_est = 1 / slope1
V2_est = 1 / slope2
ti_est = max(intercept2, 0)

# Kedalaman dari waktu intersep untuk model dua lapisan
h_est = ti_est * V1_est * V2_est / (2 * np.sqrt(max(V2_est**2 - V1_est**2, 1e-9)))

df_seis = pd.DataFrame({"Jarak (m)": x_m, "Waktu tiba simulasi (s)": t_obs})
display(df_seis)
print(f"Estimasi V1 = {V1_est:.1f} m/s")
print(f"Estimasi V2 = {V2_est:.1f} m/s")
print(f"Estimasi kedalaman antarmuka = {h_est:.2f} m")
print("Peringatan: estimasi sederhana ini sensitif terhadap pemilihan titik direct/head wave.")

plt.scatter(x_m, t_obs, label="Waktu tiba simulasi")
xx = np.linspace(x_m.min(), x_m.max(), 200)
plt.plot(xx, slope1*xx + intercept1, label="Regresi jarak pendek")
plt.plot(xx, slope2*xx + intercept2, label="Regresi jarak jauh")
plt.xlabel("Jarak (m)")
plt.ylabel("Waktu tempuh (s)")
plt.title("Kasus 1 — Kurva waktu-tempuh seismik refraksi")
plt.legend()
plt.show()

### Interpretasi dan verifikasi
- Kemiringan garis waktu-tempuh adalah kebalikan kecepatan pada segmen yang dipilih.
- Model dua lapisan mensyaratkan `V2 > V1`. Jika syarat ini tidak terpenuhi, perhitungan kedalaman tidak valid.
- Pemilihan segmen gelombang langsung dan refraksi kepala memengaruhi hasil. Untuk data nyata, gunakan analisis first-break yang lebih teliti dan evaluasi geometri lapisan.

In [ ]:
assert V1_est > 0 and V2_est > 0, "Kecepatan harus positif."
if V2_est <= V1_est:
    print("Verifikasi gagal: V2 harus lebih besar daripada V1 pada model refraksi ini.")
else:
    print("Verifikasi dasar lolos: V1 dan V2 positif serta V2 > V1.")

# KASUS 2 — GEOLISTRIK RESISTIVITAS
## Menghitung resistivitas semu konfigurasi Wenner

### Rumusan kasus
Pada konfigurasi Wenner dengan jarak elektroda berurutan sama dengan `a`, faktor geometri ideal adalah `K = 2πa`. Resistivitas semu dihitung dengan `ρa = K × ΔV/I`. Perhitungan ini tidak langsung menghasilkan resistivitas sejati bawah permukaan; interpretasi lapisan memerlukan inversi dan informasi geologi.

### Prompt AI (R-T-C-I-K-O-Q-V)
**R:** Asisten geofisika metode geolistrik.  
**T:** Hitung resistivitas semu dari beda potensial dan arus.  
**C:** Konfigurasi Wenner ideal.  
**I:** Jarak elektroda, arus, dan beda potensial.  
**K:** Pastikan satuan konsisten dan arus tidak nol.  
**O:** Tabel resistivitas semu serta grafik terhadap spasi elektroda.  
**Q:** Periksa nilai positif dan satuan ohm-meter.  
**V:** Hitung ulang satu sampel secara manual.

In [ ]:
# Data simulasi pengukuran Wenner
a_m = np.array([1, 2, 3, 4, 5, 6], dtype=float)
current_A = np.array([0.020, 0.018, 0.016, 0.015, 0.014, 0.013])
deltaV_V = np.array([0.080, 0.125, 0.155, 0.180, 0.196, 0.208])

K_wenner = 2 * np.pi * a_m
rho_app = K_wenner * deltaV_V / current_A

df_ert = pd.DataFrame({
    "Spasi a (m)": a_m,
    "Arus I (A)": current_A,
    "Beda potensial ΔV (V)": deltaV_V,
    "Faktor geometri K (m)": K_wenner,
    "Resistivitas semu (Ω m)": rho_app
})
display(df_ert)
print(f"Rata-rata resistivitas semu = {rho_app.mean():.2f} Ω m")

plt.plot(a_m, rho_app, marker="o")
plt.xlabel("Spasi elektroda a (m)")
plt.ylabel("Resistivitas semu (Ω m)")
plt.title("Kasus 2 — Resistivitas semu Wenner")
plt.show()

# Verifikasi manual untuk sampel pertama
rho_manual = 2*np.pi*a_m[0]*deltaV_V[0]/current_A[0]
print(f"Hitung manual sampel pertama = {rho_manual:.2f} Ω m")
assert np.isclose(rho_app[0], rho_manual)
assert np.all(rho_app > 0)
print("Verifikasi perhitungan lolos.")

### Interpretasi dan verifikasi
Resistivitas semu meningkat atau menurun terhadap spasi dapat mencerminkan perubahan distribusi resistivitas yang dipengaruhi kedalaman investigasi, tetapi tidak boleh langsung diterjemahkan sebagai satu lapisan tertentu. Nilai dipengaruhi kondisi kontak elektroda, heterogenitas lateral, topografi, dan noise. Untuk interpretasi bawah permukaan, lakukan inversi 2D/3D dengan pemeriksaan RMS error dan model geologi.

# KASUS 3 — GEOMAGNETIK
## Koreksi sederhana dan identifikasi anomali magnetik

### Rumusan kasus
Data magnetik lapangan dapat mengandung variasi regional dan gangguan temporal. Dalam contoh simulasi ini, anomali dihitung dengan mengurangi tren regional linier dari medan terukur. Ini adalah demonstrasi detrending sederhana, bukan pengganti koreksi diurnal, IGRF, leveling, atau reduksi ke kutub/equator jika diperlukan.

### Prompt AI (R-T-C-I-K-O-Q-V)
**R:** Asisten geofisika geomagnetik.  
**T:** Pisahkan tren regional linier dan anomali lokal.  
**C:** Profil magnetik satu dimensi.  
**I:** Posisi profil dan medan magnet simulasi.  
**K:** Nyatakan metode regional yang sederhana; jangan klaim anomali sebagai mineral tertentu.  
**O:** Tabel dan grafik medan total, regional, dan residual.  
**Q:** Pastikan residual sama dengan medan terukur dikurangi regional.  
**V:** Periksa residual rata-rata dan bentuk anomali.

In [ ]:
rng = np.random.default_rng(7)
distance_m = np.arange(0, 1001, 25, dtype=float)
regional_true = 42000 + 0.012*distance_m
local_anomaly = 80*np.exp(-((distance_m-520)/115)**2) - 25*np.exp(-((distance_m-760)/90)**2)
mag_observed = regional_true + local_anomaly + rng.normal(0, 3, len(distance_m))

# Fit tren regional linier; residual adalah anomali lokal indikatif
coef = np.polyfit(distance_m, mag_observed, 1)
mag_regional = np.polyval(coef, distance_m)
mag_residual = mag_observed - mag_regional

df_mag = pd.DataFrame({
    "Jarak (m)": distance_m,
    "Medan magnet simulasi (nT)": mag_observed,
    "Regional linier (nT)": mag_regional,
    "Residual/anomali (nT)": mag_residual
})
display(df_mag.head(10))
print(f"Rentang residual = {mag_residual.min():.2f} hingga {mag_residual.max():.2f} nT")

plt.plot(distance_m, mag_observed, label="Medan terukur simulasi", alpha=0.8)
plt.plot(distance_m, mag_regional, label="Tren regional linier")
plt.xlabel("Jarak profil (m)")
plt.ylabel("Medan magnet (nT)")
plt.title("Kasus 3a — Pemisahan regional")
plt.legend()
plt.show()

plt.axhline(0, linewidth=1)
plt.plot(distance_m, mag_residual, marker=".", label="Residual")
plt.xlabel("Jarak profil (m)")
plt.ylabel("Anomali residual (nT)")
plt.title("Kasus 3b — Anomali magnetik residual")
plt.legend()
plt.show()

assert np.allclose(mag_residual, mag_observed - mag_regional)
print("Verifikasi residual lolos.")

### Interpretasi dan verifikasi
Residual menunjukkan variasi magnetik lokal setelah tren linier dikurangi. Puncak positif atau negatif dapat berkaitan dengan kontras suseptibilitas dan geometri sumber, tetapi bukan bukti tunggal keberadaan mineral tertentu. Interpretasi memerlukan koreksi standar, parameter medan bumi, arah magnetisasi, data geologi, dan pemodelan sumber.

# KASUS 4 — GRAVITASI
## Mengestimasi anomali gravitasi lempeng sederhana

### Rumusan kasus
Untuk ilustrasi awal, anomali gravitasi ideal dari lempeng horizontal tak berhingga dapat ditulis `Δg = 2πG Δρ t`, dengan `G` konstanta gravitasi, `Δρ` kontras densitas, dan `t` ketebalan. Rumus ini hanya berlaku untuk geometri ideal tersebut. Data survei nyata membutuhkan koreksi drift, pasang surut, lintang, free-air, Bouguer, dan terrain sesuai kebutuhan.

### Prompt AI (R-T-C-I-K-O-Q-V)
**R:** Asisten geofisika gravitasi.  
**T:** Hitung anomali gravitasi ideal dari kontras densitas dan ketebalan.  
**C:** Model lempeng horizontal tak berhingga.  
**I:** Kontras densitas dan ketebalan.  
**K:** Gunakan SI dan ubah m/s² ke mGal (`1 mGal = 10⁻⁵ m/s²`).  
**O:** Hasil numerik dan grafik sensitivitas.  
**Q:** Periksa dimensi dan tanda anomali.  
**V:** Hitung ulang dengan satu parameter dan jelaskan keterbatasan geometri.

In [ ]:
G = 6.67430e-11  # m^3 kg^-1 s^-2
thickness_m = np.linspace(100, 1500, 50)
density_contrast = 300.0  # kg/m^3, simulasi
dg_ms2 = 2 * np.pi * G * density_contrast * thickness_m
dg_mgal = dg_ms2 / 1e-5

df_grav = pd.DataFrame({
    "Ketebalan (m)": thickness_m,
    "Anomali gravitasi ideal (mGal)": dg_mgal
})
display(df_grav.iloc[::10].reset_index(drop=True))
print(f"Anomali pada ketebalan 800 m = {(2*np.pi*G*density_contrast*800)/1e-5:.4f} mGal")

plt.plot(thickness_m, dg_mgal)
plt.xlabel("Ketebalan lempeng (m)")
plt.ylabel("Anomali gravitasi ideal (mGal)")
plt.title("Kasus 4 — Sensitivitas anomali terhadap ketebalan")
plt.show()

# Pemeriksaan nilai dan linearitas model ideal
assert np.all(dg_mgal > 0)
assert np.isclose(dg_mgal[-1]/dg_mgal[0], thickness_m[-1]/thickness_m[0])
print("Verifikasi model ideal lolos: anomali berbanding lurus dengan ketebalan.")

### Interpretasi dan verifikasi
Pada model lempeng ideal dengan kontras densitas positif, anomali meningkat linier terhadap ketebalan. Bentuk dan amplitudo anomali nyata juga dipengaruhi geometri, kedalaman, densitas, dan distribusi tiga dimensi sumber. Karena itu, rumus ini cocok untuk latihan sensitivitas, bukan untuk mengklaim kedalaman atau densitas target dari data lapangan secara unik.

# KASUS 5 — MAGNETOTELURIK (MT)
## Menghitung resistivitas semu dari impedansi

### Rumusan kasus
Dalam konvensi SI yang umum, resistivitas semu MT dapat dihitung dari amplitudo impedansi menggunakan `ρa = |Z|²/(μ0 ω)`, dengan `ω = 2πf` dan `μ0 = 4π × 10⁻⁷ H/m`. Kedalaman investigasi tidak dapat ditentukan secara unik dari frekuensi saja; ia dipengaruhi resistivitas bawah permukaan dan struktur geologi.

### Prompt AI (R-T-C-I-K-O-Q-V)
**R:** Asisten geofisika magnetotelurik.  
**T:** Hitung resistivitas semu untuk beberapa frekuensi.  
**C:** Data impedansi kompleks berasal dari simulasi.  
**I:** Frekuensi dan komponen impedansi.  
**K:** Gunakan satuan SI, frekuensi positif, dan definisi resistivitas semu yang jelas.  
**O:** Tabel dan kurva resistivitas semu terhadap frekuensi.  
**Q:** Periksa nilai positif dan konsistensi satuan.  
**V:** Verifikasi satu frekuensi dengan hitungan manual; jelaskan bahwa kurva bukan inversi.

In [ ]:
mu0 = 4*np.pi*1e-7  # H/m
freq_hz = np.logspace(-2, 2, 25)  # 0.01–100 Hz

# Impedansi kompleks simulasi yang dibuat dari target rho_app bervariasi
rho_target = 30 + 400/(1 + (freq_hz/0.7)**1.2)  # ohm m, hanya untuk membentuk contoh
omega = 2*np.pi*freq_hz
Z_abs = np.sqrt(rho_target * mu0 * omega)
phase_deg = np.full_like(freq_hz, 45.0)
Z_complex = Z_abs * (np.cos(np.deg2rad(phase_deg)) + 1j*np.sin(np.deg2rad(phase_deg)))

rho_mt = np.abs(Z_complex)**2/(mu0*omega)
df_mt = pd.DataFrame({
    "Frekuensi (Hz)": freq_hz,
    "|Z| (Ω)": np.abs(Z_complex),
    "Fase (derajat)": np.angle(Z_complex, deg=True),
    "Resistivitas semu (Ω m)": rho_mt
})
display(df_mt)
plt.semilogx(freq_hz, rho_mt, marker="o")
plt.xlabel("Frekuensi (Hz)")
plt.ylabel("Resistivitas semu (Ω m)")
plt.title("Kasus 5 — Kurva resistivitas semu MT (simulasi)")
plt.show()

i = 10
rho_manual = np.abs(Z_complex[i])**2/(mu0*2*np.pi*freq_hz[i])
print(f"Verifikasi manual pada f={freq_hz[i]:.3f} Hz: {rho_manual:.2f} Ω m")
assert np.all(freq_hz > 0)
assert np.all(rho_mt > 0)
assert np.allclose(rho_mt, rho_target)
print("Verifikasi resistivitas semu MT lolos.")

### Interpretasi dan verifikasi
Pada data simulasi ini, resistivitas semu berubah terhadap frekuensi sesuai kurva yang sengaja dibentuk untuk latihan. Dalam survei nyata, kurva MT dipengaruhi struktur resistivitas 1D/2D/3D, noise, static shift, orientasi tensor impedansi, dan kualitas pengukuran. Interpretasi geologi umumnya memerlukan pemeriksaan data, analisis tensor, serta inversi yang sesuai.

# 6. Ringkasan Hasil dan Evaluasi

Bagian berikut merangkum keluaran numerik utama dari lima kasus. Nilai yang muncul berasal dari data simulasi dan parameter asumsi pada notebook ini.

In [ ]:
summary = pd.DataFrame([
    {"Metode": "Seismik refraksi", "Parameter utama": "V1, V2, kedalaman antarmuka", "Status": "Estimasi regresi sederhana"},
    {"Metode": "Geolistrik Wenner", "Parameter utama": "Resistivitas semu", "Status": "Dihitung dari K × ΔV/I"},
    {"Metode": "Geomagnetik", "Parameter utama": "Anomali residual", "Status": "Tren regional linier dikurangi"},
    {"Metode": "Gravitasi", "Parameter utama": "Anomali lempeng ideal", "Status": "Model analitik sederhana"},
    {"Metode": "Magnetotelurik", "Parameter utama": "Resistivitas semu", "Status": "Dihitung dari impedansi simulasi"}
])
display(summary)

## 7. Kesimpulan

1. Prompt AI yang baik harus mendefinisikan peran, tugas, konteks, input, batasan, keluaran, kualitas, dan verifikasi.
2. Python dapat membantu melakukan perhitungan berulang, membuat tabel, memvisualisasikan data, serta memeriksa konsistensi hasil.
3. Hasil numerik tidak otomatis menjadi interpretasi geologi yang benar. Asumsi model, kualitas data, geometri sumber, dan konteks geologi tetap harus dievaluasi.
4. Semua angka pada notebook ini merupakan **data simulasi untuk latihan**, bukan hasil pengukuran lapangan.

## 8. Referensi dasar

- Reynolds, J. M. (2011). *An Introduction to Applied and Environmental Geophysics* (2nd ed.). Wiley-Blackwell.
- Telford, W. M., Geldart, L. P., & Sheriff, R. E. (1990). *Applied Geophysics* (2nd ed.). Cambridge University Press.
- Chave, A. D., & Jones, A. G. (Eds.). (2012). *The Magnetotelluric Method: Theory and Practice*. Cambridge University Press.

**Catatan akademik:** sesuaikan pembahasan dengan arahan dosen dan tambahkan sumber yang diwajibkan dalam perkuliahan.